In [ ]:
import os 
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
import torch.nn as nn
from torch.nn import functional as F
from torch import optim
import torchaudio.transforms as T
import matplotlib.pyplot as plt
from transformers import Wav2Vec2CTCTokenizer,get_cosine_schedule_with_warmup
from jiwer import wer 
import torchaudio

In [ ]:
tokenizer = Wav2Vec2CTCTokenizer.from_pretrained("facebook/wave2vec2-base")

In [ ]:
class SpeechDataset(Dataset):
    def __init__(self, data_dir, include_splits= ["train-clean-100", "train-clean-360", "train-clean-500"],sampling_rate=16000, num_audio_channels=1):
        
        if isinstance(include_splits, str):
            include_splits = [include_splits]
            
        self.sampling_rate = sampling_rate
        self.num_audio_channels = num_audio_channels
        
        # path to audio files
        self.librispeech_data = []
        for split in include_splits:
            split_path = os.path.join(data_dir, split)
            
            for speaker in os.listdir(split_path):
                path_to_speaker = os.path.join(split_path, speaker)
                
                
                for section in os.listdir(split_path):
                    path_to_section = os.path.join(path_to_speaker, section)
                    
                    # split flac audio and text transcripts
                    files = os.listdir(path_to_section)
                    transcript_file = [path for path in files if ".txt" in path[0]]
                    
                    # load transcript 
                    with open(os.path.join(path_to_section, transcript_file), "r") as f:
                        transcripts = f.readlines()
                        
                    # split transcript by audio filename 
                    for line in transcripts:
                        split_line = line.split()
                        audio_root = split_line[0]
                        audio_file = audio_root + ".flac"
                        full_path_to_audiofile = os.path.join(path_to_section, audio_file)
                        transcript = " ".join(split_line[1:]).strip()
                        
                        self.librispeech_data.append((full_path_to_audiofile, transcript))
                        
                        
        self.audio_to_mels = T.MelSpectrogram(sample_rate=self.sampling_rate, n_mels=80)
        self.amp_to_db = T.AmplitudeToDB(top_db=80.0)
        
    def __len__(self):
        return len(self.librispeech_data)
    
    def __getitem__(self, idx):
        
        # path to audio and transcript
        audio_path , transcript = self.librispeech_data[idx]
        
        audio,original_sr = torchaudio.load(audio_path, normalize=True)
        
        if original_sr != self.sampling_rate:
            resampler = T.Resample(orig_freq=original_sr, new_freq=self.sampling_rate)
            audio = resampler(audio)
        
        # create melspec
        mel = self.audio_to_mels(audio)
        
        # to decibles
        mel_db = self.amp_to_db(mel)
        
        # normalise melspec
        mel = (mel - mel.mean()) / (mel.std() + 1e-6)
        
        # tokenize
        tokenized_transcript = torch.tensor(tokenizer.encode(transcript))
        
        # transpose mel spec so when we pad collate_fun we pad on the time axis
        sample = {"input_values": mel[0].T, "labels": tokenized_transcript}
        return sample


dataset = SpeechDataset(data_dir="", include_splits="train-clean-100")
sample = next(iter(dataset))

plt.figure(figsize=(15,5))
plt.imshow(sample["input_values"].T)
plt.axis("off")
plt.gca().invert_yaxis()
plt.show()

# collate function 

In [ ]:
def collate_fn(batch):

    # attention masks, sub_attention_masks, span_masks and our sampled negatives!

    # Sort Batch from longest to shortest (for future packed padding) 
    batch = sorted(batch, key=lambda x: x["input_values"].shape[0], reverse=True)
    
    # audios from our Batch Dict
    batch_mels = [sample["input_values"] for sample in batch]
    batch_transcripts = [sample["labels"] for sample in batch]

    # len of audios
    seq_lens = torch.tensor([b.shape[0] for b in batch_mels], dtype=torch.long)

    #  Pad and stack specs
    spectrograms = torch.nn.utils.rnn.pad_sequence(batch_mels, batch_first=True, padding_value=0)

    # convert to (B x C x H x W)
    spectrograms = spectrograms.unsqueeze(1).transpose(-1,-2)

   
    target_lengths = torch.tensor([len(t) for t in batch_transcripts], dtype=torch.long)

    ### Pack Transcripts (CTC Loss Can Take Packed Targets) ###
    packed_transcripts = torch.cat(batch_transcripts)

    # create batch
    batch = {"input_values": spectrograms, 
             "seq_lens": seq_lens, 
             "labels": packed_transcripts, 
             "target_lengths": target_lengths}

    return batch

# Test Collate fun
loader = DataLoader(dataset, batch_size=5, collate_fn=collate_fn)
batch = next(iter(loader))

print("Input Values:", batch["input_values"].shape)
print("Seq Lens", batch["seq_lens"])
print("Labels:", batch["labels"].shape)
print("Target Lengths:", batch["target_lengths"])

# As required by the CTC loss, sum of the target lengths must equal the leng1276th of the flattened labels 
if batch["target_lengths"].sum() == len(batch["labels"]):
    print("Sucess, Same Length")